# Verificación de potencia estadística — Plantilla

**Universidad Nacional Abierta y a Distancia (UNAD)**
Doctorado en Tecnologías de Información · Entorno 203538899
**Ciclo 1 · Fase 1 — Lectura crítica · Semana 3**

---

## ¿Para qué sirve esta plantilla?

En el análisis metodológico del artículo seleccionado, esta plantilla te permite **verificar
si el tamaño muestral del estudio era suficiente** para detectar el tamaño del efecto
reportado, usando `statsmodels.stats.power`. Es la **Sección 2** del Reporte de Análisis
Metodológico (6–8 páginas, APA 7).

### Cómo usarla

1. Completa el bloque **PARÁMETROS DEL ARTÍCULO** (más abajo) con los datos del estudio que analizas.
2. Ve a **Kernel → Restart & Run All**.
3. Lee las salidas y **responde las preguntas de interpretación** en las celdas Markdown marcadas con ✍️.
4. Adjunta este notebook (`.ipynb`) reproducible a tu reporte.

> **Convenciones del entorno (Cohen, 1988):** α = 0.05, potencia (1−β) = 0.80. Si el
> artículo **no reporta** el tamaño del efecto, usa un efecto *medium* convencional según la
> prueba (d = 0.5 para dos grupos; f = 0.25 para ANOVA; f² = 0.15 para regresión).

## Paso 0 · Importar librerías

In [ ]:
# Librerías del cálculo de potencia
import numpy as np
from scipy import stats
from statsmodels.stats.power import (
    TTestIndPower,      # dos grupos independientes (d de Cohen)
    FTestAnovaPower,    # ANOVA de un factor (f de Cohen)
)

# Potencia de regresión múltiple con la distribución F no central (convención de Cohen).
# statsmodels.FTestPower usa una parametrización de no-centralidad distinta a f² de Cohen,
# por eso calculamos la potencia de la prueba F de forma explícita: lambda = f² * n.
def power_regresion(f2, n, p, alpha=0.05):
    df_num = p            # nº de predictores
    df_denom = n - p - 1  # gl del error
    if df_denom <= 0:
        return 0.0
    ncp = f2 * n                                   # no-centralidad (Cohen)
    f_crit = stats.f.ppf(1 - alpha, df_num, df_denom)
    return 1 - stats.ncf.cdf(f_crit, df_num, df_denom, ncp)

def n_requerido_regresion(f2, p, alpha=0.05, power=0.80, n_max=100000):
    for n in range(p + 2, n_max):
        if power_regresion(f2, n, p, alpha) >= power:
            return n
    return None

print("Librerías de potencia importadas correctamente.")

## Paso 1 · PARÁMETROS DEL ARTÍCULO

**Edita solo esta celda.** Describe el estudio que analizaste. Deja en `None` lo que no
aplique a tu caso; cada paso usa únicamente lo que necesita.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARÁMETROS DEL ARTÍCULO ANALIZADO  ·  (edita estos valores)
# ─────────────────────────────────────────────────────────────────────────────

# Identificación (para el encabezado del informe)
ARTICULO   = "Apellido, A. (Año). Título del artículo. Revista."   # referencia breve
LINEA_DTI  = "CRIC"          # una de: CRIC · IASC · IoTCI · ADVTD

# Convenciones del entorno (normalmente NO se cambian)
ALFA    = 0.05               # nivel de significancia
POTENCIA = 0.80              # potencia deseada (1 - beta)

# Tamaño muestral REAL reportado en el artículo
N_TOTAL = 120                # n total del estudio
N_GRUPOS = 2                 # nº de grupos comparados (para t-test/ANOVA)

# Tamaño del efecto REPORTADO por el artículo (None si no lo reporta)
#   - dos grupos  -> d de Cohen        (medium convencional = 0.5)
#   - ANOVA       -> f de Cohen        (medium convencional = 0.25)
#   - regresión   -> f² de Cohen       (medium convencional = 0.15)
EFECTO_REPORTADO = None      # p. ej. 0.42 ; si es None se usa el medium convencional

# Para la Sección de regresión (Paso 3C), si aplica:
N_PREDICTORES = None         # nº de predictores del modelo (None si no hay regresión)

print("Parámetros cargados:")
print(f"  Artículo   : {ARTICULO}")
print(f"  Línea DTI  : {LINEA_DTI}")
print(f"  n total    : {N_TOTAL}  ·  grupos: {N_GRUPOS}")
print(f"  alfa={ALFA} · potencia objetivo={POTENCIA}")
print(f"  efecto reportado: {EFECTO_REPORTADO if EFECTO_REPORTADO is not None else 'no reportado (se usará medium convencional)'}")

## Paso 2 · Elige el tipo de prueba del estudio

Ejecuta **solo el sub-paso que corresponda** a la prueba principal del artículo:

- **3A** — Comparación de **dos grupos** (t-test independiente) → efecto *d* de Cohen.
- **3B** — Comparación de **tres o más grupos** (ANOVA de un factor) → efecto *f* de Cohen.
- **3C** — **Regresión múltiple** → efecto *f²* de Cohen.

Cada sub-paso calcula dos cosas: (1) la **potencia alcanzada** con el *n* real del artículo, y
(2) el **_n_ requerido** para alcanzar potencia = 0.80.

### Paso 3A · Dos grupos — t-test independiente (d de Cohen)

In [ ]:
# Ejecuta esta celda SOLO si el artículo compara dos grupos con t-test.
d = EFECTO_REPORTADO if EFECTO_REPORTADO is not None else 0.5   # medium = 0.5
n_por_grupo = N_TOTAL / 2

analisis = TTestIndPower()

# (1) Potencia alcanzada con el n real
potencia_real = analisis.power(effect_size=d, nobs1=n_por_grupo,
                               alpha=ALFA, ratio=1.0, alternative="two-sided")

# (2) n por grupo requerido para potencia = 0.80
n_req_por_grupo = analisis.solve_power(effect_size=d, alpha=ALFA, power=POTENCIA,
                                       ratio=1.0, alternative="two-sided")

print(f"t-test independiente · d = {d} ({'reportado' if EFECTO_REPORTADO is not None else 'medium convencional'})")
print(f"  n por grupo (real)         : {n_por_grupo:.0f}")
print(f"  Potencia alcanzada (real)  : {potencia_real:.3f}")
print(f"  n por grupo requerido      : {np.ceil(n_req_por_grupo):.0f}  (n total = {np.ceil(n_req_por_grupo)*2:.0f})")
print()
print("  ✅ Muestra suficiente" if potencia_real >= POTENCIA
      else "  ⚠️  Muestra insuficiente para el efecto y potencia objetivo")

### Paso 3B · Tres o más grupos — ANOVA de un factor (f de Cohen)

In [ ]:
# Ejecuta esta celda SOLO si el artículo compara 3+ grupos con ANOVA.
f = EFECTO_REPORTADO if EFECTO_REPORTADO is not None else 0.25   # medium = 0.25
n_por_grupo = N_TOTAL / N_GRUPOS

analisis = FTestAnovaPower()

potencia_real = analisis.power(effect_size=f, nobs=N_TOTAL,
                               alpha=ALFA, k_groups=N_GRUPOS)

n_total_req = analisis.solve_power(effect_size=f, alpha=ALFA, power=POTENCIA,
                                   k_groups=N_GRUPOS)

print(f"ANOVA de un factor · f = {f} ({'reportado' if EFECTO_REPORTADO is not None else 'medium convencional'}) · k = {N_GRUPOS} grupos")
print(f"  n total (real)             : {N_TOTAL}   ({n_por_grupo:.0f} por grupo)")
print(f"  Potencia alcanzada (real)  : {potencia_real:.3f}")
print(f"  n total requerido          : {np.ceil(n_total_req):.0f}  ({np.ceil(n_total_req/N_GRUPOS):.0f} por grupo)")
print()
print("  ✅ Muestra suficiente" if potencia_real >= POTENCIA
      else "  ⚠️  Muestra insuficiente para el efecto y potencia objetivo")

### Paso 3C · Regresión múltiple (f² de Cohen)

In [ ]:
# Ejecuta esta celda SOLO si el artículo ajusta una regresión múltiple.
# f² de Cohen: 0.02 (small), 0.15 (medium), 0.35 (large).
f2 = EFECTO_REPORTADO if EFECTO_REPORTADO is not None else 0.15   # medium = 0.15

if N_PREDICTORES is None:
    print("Define N_PREDICTORES en el Paso 1 para usar este sub-paso.")
else:
    potencia_real = power_regresion(f2, N_TOTAL, N_PREDICTORES, alpha=ALFA)
    n_req = n_requerido_regresion(f2, N_PREDICTORES, alpha=ALFA, power=POTENCIA)

    print(f"Regresión múltiple · f² = {f2} ({'reportado' if EFECTO_REPORTADO is not None else 'medium convencional'}) · {N_PREDICTORES} predictores")
    print(f"  n (real)                   : {N_TOTAL}")
    print(f"  Potencia alcanzada (real)  : {potencia_real:.3f}")
    print(f"  n requerido                : {n_req}")
    print()
    print("  ✅ Muestra suficiente" if potencia_real >= POTENCIA
          else "  ⚠️  Muestra insuficiente para el efecto y potencia objetivo")

## Paso 4 · Curva de potencia (opcional, recomendado para el reporte)

Visualiza cómo crece la potencia con el tamaño muestral, marcando el *n* real del artículo.
Útil como figura de la Sección 2. Ajusta la variable `TIPO` a `"ttest"`, `"anova"` o
`"regresion"` según el estudio.

In [ ]:
import matplotlib.pyplot as plt

TIPO = "ttest"   # "ttest" · "anova" · "regresion"

fig, ax = plt.subplots(figsize=(8, 5))

if TIPO == "ttest":
    d = EFECTO_REPORTADO if EFECTO_REPORTADO is not None else 0.5
    ns = np.arange(5, 200, 2)
    pot = TTestIndPower().power(effect_size=d, nobs1=ns, alpha=ALFA, ratio=1.0)
    ax.plot(ns, pot, color="#1c4587", lw=2)
    ax.axvline(N_TOTAL/2, ls="--", color="#cc0000", label=f"n real por grupo = {N_TOTAL/2:.0f}")
    ax.set_xlabel("n por grupo")
    titulo = f"Curva de potencia · t-test · d = {d}"
elif TIPO == "anova":
    f = EFECTO_REPORTADO if EFECTO_REPORTADO is not None else 0.25
    ns = np.arange(N_GRUPOS*3, 400, 3)
    pot = FTestAnovaPower().power(effect_size=f, nobs=ns, alpha=ALFA, k_groups=N_GRUPOS)
    ax.plot(ns, pot, color="#1c4587", lw=2)
    ax.axvline(N_TOTAL, ls="--", color="#cc0000", label=f"n total real = {N_TOTAL}")
    ax.set_xlabel("n total")
    titulo = f"Curva de potencia · ANOVA · f = {f} · k = {N_GRUPOS}"
else:  # regresion
    f2 = EFECTO_REPORTADO if EFECTO_REPORTADO is not None else 0.15
    p = N_PREDICTORES if N_PREDICTORES is not None else 3
    ns = np.arange(p+5, 400, 3)
    pot = np.array([power_regresion(f2, n, p, alpha=ALFA) for n in ns])
    ax.plot(ns, pot, color="#1c4587", lw=2)
    ax.axvline(N_TOTAL, ls="--", color="#cc0000", label=f"n real = {N_TOTAL}")
    ax.set_xlabel("n total")
    titulo = f"Curva de potencia · regresión · f² = {f2} · {p} predictores"

ax.axhline(POTENCIA, ls=":", color="gray", label=f"potencia objetivo = {POTENCIA}")
ax.set_ylabel("Potencia (1 − β)")
ax.set_title(titulo)
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Paso 5 · Interpretación ✍️

**Responde en esta celda** (doble clic para editar). Sustenta cada respuesta en los
resultados anteriores y en los referentes del entorno (Cohen, 1988; Wasserstein & Lazar,
2016). Esta interpretación es el núcleo de la Sección 2 del reporte.

1. **¿El estudio tenía muestra suficiente?** Compara la potencia alcanzada con el *n* real
   frente al objetivo de 0.80. Si fue insuficiente, ¿cuál habría sido el *n* necesario?

   > _(tu respuesta aquí)_

2. **¿Las conclusiones del estudio son estadísticamente robustas?** ¿El riesgo de error
   Tipo II (no detectar un efecto real por falta de potencia) compromete las conclusiones de
   los autores?

   > _(tu respuesta aquí)_

3. **Tamaño del efecto vs. valor p.** ¿El artículo reporta tamaño del efecto e intervalos de
   confianza, o se apoya solo en el valor p? Relaciona con Wasserstein & Lazar (2016).

   > _(tu respuesta aquí)_

4. **Conexión con la línea DTI.** ¿Qué implica este hallazgo de potencia para la calidad de
   la evidencia en la línea {LINEA_DTI}?

   > _(tu respuesta aquí)_